In [1]:
import os
import json
import csv
from openai import OpenAI
import pandas as pd
import time

In [ ]:
client = OpenAI(
)

In [ ]:
# Cheapest model
models = [
    "gpt-3.5-turbo-instruct",
    "gpt-4o",
    "gpt-4o-mini",
    "o1-2024-12-17",
    "o3-mini-2025-01-31"
]
results_filename = "../data/results.csv"
latency_filename = "../data/latency_results.csv"

In [4]:
def export_model(file_name, data, headers):
    with open(file_name, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(headers)  # Write the header row
        writer.writerows(data)  # Write all rows of data

In [5]:
def load_scenarios(file_path):
    with open(file_path, "r") as file:
        scenarios = json.load(file)
    return scenarios["Scenarios"]
scenario_filepath = file_path + "scenarios.json"
scenarios = load_scenarios(scenario_filepath)

In [6]:
def load_scenic_scenarios(directory: str):
    scenarios = []
    
    # Ensure the directory exists
    if not os.path.isdir(directory):
        raise ValueError(f"The directory '{directory}' does not exist.")
    
    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".scenic"):
            file_path = os.path.join(directory, filename)
            
            # Read file contents
            with open(file_path, "r", encoding="utf-8") as file:
                scenarios.append(file.read())
    
    return scenarios

In [7]:
prompt_instructions = """
You are an AI driving analysis tool. Your primary function is to assess whether a given driving scenario is consistent or exhibits signs of an adversarial attack (e.g., tampered traffic signs). You have expert knowledge of traffic regulations, road signage, and common driving practices.

Given a driving scenario in JSON format, your goal is to:
    1. Determine if there is any inconsistency in the scenario (e.g., a sign or speed limit that conflicts with typical real-world traffic laws).
    2. Provide a clear and concise explanation of your reasoning, identifying which specific part of the scenario is inconsistent and referencing relevant traffic rules or logical deductions.

Use the following exact response format. Fill in the placeholders appropriately:

    Inconsistency Detected: [Yes/No]
    Reasoning: [Provide a thorough and concise explanation. Keep it to 3-5 sentences and ensure it demonstrates your reasoning.]

Follow a step-by-step reasoning process internally but only provide the final output in the specified format. Be precise and adhere strictly to the format.
"""

In [8]:
scenic_prompt_instructions = """
You are an AI driving analysis tool. Your primary function is to assess whether a given driving scenario is consistent or exhibits signs of an adversarial attack (e.g., tampered traffic signs). You have expert knowledge of traffic regulations, road signage, and common driving practices.

Given a driving scenario in the Scenic Programming Language, your goal is to:
    1. Determine if there is any inconsistency in the scenario (e.g., a sign or speed limit that conflicts with typical real-world traffic laws).
    2. Provide a clear and concise explanation of your reasoning, identifying which specific part of the scenario is inconsistent and referencing relevant traffic rules or logical deductions.

Use the following exact response format. Fill in the placeholders appropriately:

    Inconsistency Detected: [Yes/No]
    Reasoning: [Provide a thorough and concise explanation. Keep it to 3-5 sentences and ensure it demonstrates your reasoning.]

Follow a step-by-step reasoning process internally but only provide the final output in the specified format. Be precise and adhere strictly to the format.
"""

In [9]:
scenario_filepath = file_path + "scenarios.json"
scenarios = load_scenarios(scenario_filepath)
scenic_scenarios = load_scenic_scenarios(scenic_scenarios_file_path)

In [10]:
# Model parameters
'''
From: https://community.openai.com/t/cheat-sheet-mastering-temperature-and-top-p-in-chatgpt-api/172683
With these values GPT4 generates creative and diverse text for storytelling
'''

TEMP = 1.5
TOP_P = 0.9
TOP_K = 40
PRESENCE_PENALTY = 0.0
FREQUENCY_PENALTY = 0.0
MAX_TOKENS = 256

In [11]:
# For scenarios in json format
#for model_name in models: 
#    
#    outputs = []
#    latency_outputs = []
#    print()
#    
#    for scenario in scenarios:
#        scenario_input =(
#            f"Scenario ID: {scenario['id']}\n"
#            f"Road Type: {scenario['RoadType']}\n"
#            f"Speed: {scenario['Speed']} mph\n"
#            f"Lane Markings: {', '.join(scenario['LaneMarkings'])}\n"
#            f"Traffic Signs: {', '.join(scenario['TrafficSigns'])}\n"
#            f"{', '.join([str(device) for device in scenario['TrafficControlDevices']]) if scenario['TrafficControlDevices'] else 'None'}\n"
#            f"Time of Day: {scenario['TimeOfDay']}\n"
#            f"Description: {scenario['Description']}\n"
#        )
#        print(f"For model: {model_name} - Processing Scenario {scenario['id']}...")
#        prompt = f"{prompt_instructions}\n{scenario_input}"
#
#        start_time = time.time()
#        output_text = ""
#        
#        if "4o" in model_name:
#            response = client.chat.completions.create(
#                model = model_name,
#                temperature=TEMP,
#                top_p = TOP_P,
#                presence_penalty = PRESENCE_PENALTY,
#                frequency_penalty = FREQUENCY_PENALTY,
#                max_tokens = MAX_TOKENS,
#                messages=[
#                    #{"role": "system", "content": prompt_instructions},
#                    {"role": "system", "content": scenic_prompt_instructions},
#                    {"role": "user", "content": json.dumps(scenario)}
#                ],
#            )
#            output_text = response.choices[0].message.content.strip()
#        elif "3.5" in model_name: #3.5
#            response = client.completions.create(
#                model = model_name,
#                temperature=TEMP,
#                top_p = TOP_P,
#                presence_penalty = PRESENCE_PENALTY,
#                frequency_penalty = FREQUENCY_PENALTY,
#                max_tokens = MAX_TOKENS,
#                prompt = prompt
#            )
#            output_text = response.choices[0].text.strip()
#        else:
#            response = client.chat.completions.create(
#                model = model_name,
#                presence_penalty = PRESENCE_PENALTY,
#                frequency_penalty = FREQUENCY_PENALTY,
#                messages=[
#                    {"role": "system", "content": prompt_instructions},
#                    {"role": "user", "content": json.dumps(scenario)}
#                ],
#            )
#            output_text = response.choices[0].message.content.strip()
#            
#        outputs.append(output_text)
#        
#        elapsed_time = round(time.time() - start_time, 4)
#        latency_outputs.append(elapsed_time)
#    
#    ## Output results
#    # read current csv file 
#    current_results_df = pd.read_csv(results_filename)
#
#    scenario_id = "scenario_id"
#    new_results_df = pd.DataFrame({
#        scenario_id: [scenario["id"] for scenario in scenarios],
#        model_name: outputs
#    })
#
#    # Merge new model's ouputs with existing DataFrames
#    updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")
#
#    updated_df.to_csv(results_filename, index=False)
#    
#    
#    ## Output Latency results
#    #read current latency csv file 
#    current_latency_results_df = pd.read_csv(latency_filename)
#
#    new_latency_df = pd.DataFrame({
#        scenario_id: [scenario["id"] for scenario in scenarios],
#        f"latency_{model_name}": latency_outputs
#
#    })
#
#    # Merge new model's ouputs with existing DataFrames
#    updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")
#
#    updated_latency_df.to_csv(latency_filename, index=False)
#
## export_to_csv(results_file_name, outputs, csv_headers)

In [13]:
# For scenarios in txt format
for model_name in models: 
    
    outputs = []
    latency_outputs = []
    print()
    
    for index, scenario in enumerate(scenic_scenarios):
        print(f"For model: {model_name} - Processing Scenario {index}...")
        prompt = f"{prompt_instructions}\n{scenario}"

        start_time = time.time()
        output_text = ""
        
        if "4o" in model_name:
            response = client.chat.completions.create(
                model = model_name,
                temperature=TEMP,
                top_p = TOP_P,
                presence_penalty = PRESENCE_PENALTY,
                frequency_penalty = FREQUENCY_PENALTY,
                max_tokens = MAX_TOKENS,
                messages=[
                    #{"role": "system", "content": prompt_instructions},
                    {"role": "system", "content": scenic_prompt_instructions},
                    #{"role": "user", "content": json.dumps(scenario)}
                    {"role": "user", "content": scenario}
                ],
            )
            output_text = response.choices[0].message.content.strip()
        elif "3.5" in model_name: #3.5
            response = client.completions.create(
                model = model_name,
                temperature=TEMP,
                top_p = TOP_P,
                presence_penalty = PRESENCE_PENALTY,
                frequency_penalty = FREQUENCY_PENALTY,
                max_tokens = MAX_TOKENS,
                prompt = prompt
            )
            output_text = response.choices[0].text.strip()
        else:
            response = client.chat.completions.create(
                model = model_name,
                presence_penalty = PRESENCE_PENALTY,
                frequency_penalty = FREQUENCY_PENALTY,
                messages=[
                    {"role": "system", "content": prompt_instructions},
                    #{"role": "user", "content": json.dumps(scenario)}
                    {"role": "user", "content": scenario}
                ],
            )
            output_text = response.choices[0].message.content.strip()
            
        outputs.append(output_text)
        
        elapsed_time = round(time.time() - start_time, 4)
        latency_outputs.append(elapsed_time)
    
    ## Output results
    # read current csv file 
    current_results_df = pd.read_csv(results_filename)

    #print("len(outputs)", len(outputs))
    #print("len([index for index, _ in enumerate(scenarios, start=1)])", len([index for index, _ in enumerate(scenarios, start=1)]))
    scenario_id = "scenario_id"
    
    new_results_df = pd.DataFrame({
        scenario_id: [index for index, _ in enumerate(scenic_scenarios, start=1)],
        model_name: outputs
    })

    # Merge new model's ouputs with existing DataFrames
    updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")

    updated_df.to_csv(results_filename, index=False)
    
    
    ## Output Latency results
    #read current latency csv file 
    current_latency_results_df = pd.read_csv(latency_filename)

    new_latency_df = pd.DataFrame({
        scenario_id: [index for index, _ in enumerate(scenic_scenarios, start=1)],
        #scenario_id: [scenario["id"] for scenario in scenarios],
        f"latency_{model_name}": latency_outputs

    })

    # Merge new model's ouputs with existing DataFrames
    updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")

    updated_latency_df.to_csv(latency_filename, index=False)

# export_to_csv(results_file_name, outputs, csv_headers)


For model: gpt-3.5-turbo-instruct - Processing Scenario 0...
For model: gpt-3.5-turbo-instruct - Processing Scenario 1...
For model: gpt-3.5-turbo-instruct - Processing Scenario 2...


KeyboardInterrupt: 

In [ ]:
for response in outputs:
    print(response)
    print()